# Vincio: the data & analytics plane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ohswedd/vincio/blob/main/examples/notebooks/05_data_analysis.ipynb)

A data question over a table should not pour every row into the prompt. In Vincio it becomes a
**read-only-verified query** that runs where the data lives, returns an answer cited to the **exact
source cells**, and **re-derives from the bytes** (`verify`). This notebook walks the analyst rung
end to end, fully offline on the standard-library SQL engine:

- **(A) `app.query_data`** — a natural-language question becomes a cited, verifiable query;
- **(B) `app.analyze_data`** — a bounded, multi-step exploration becomes a cited narrative;
- **(C) `app.generate_chart`** — a cited result becomes a content-bound, data-bound Vega-Lite chart.

Everything runs with **no network and no API keys** thanks to the bundled mock provider.

> **To run against a real model, set `VINCIO_PROVIDER` and the matching API key**
> (e.g. `VINCIO_PROVIDER=openai`, `VINCIO_MODEL=gpt-4o-mini`, `OPENAI_API_KEY=...`). The data plane
> itself is model-free; the provider only powers downstream natural-language answers.

In [ ]:
!pip install -q vincio

## Setup: an offline-first provider

A bare `ContextApp()` defaults to OpenAI and needs an API key. The helper below returns the
deterministic **mock provider** by default, so the whole notebook runs offline; set
`VINCIO_PROVIDER` to swap in a real model with no other code changes.

In [ ]:
import os

from vincio import ContextApp
from vincio.providers import MockProvider, build_provider


def _provider():
    """Offline mock by default; a real provider when VINCIO_PROVIDER is set."""
    name = os.environ.get("VINCIO_PROVIDER", "mock")
    if name == "mock":
        return MockProvider(), "mock-1"
    return build_provider(name), os.environ.get("VINCIO_MODEL", "gpt-4o-mini")


provider, model = _provider()
app = ContextApp(name="analytics", provider=provider, model=model)
print(f"provider ready: model={model!r}")

## Register a dataset

`app.register_dataset(rows, columns=..., name=...)` ingests a small inline table into the app's
**data catalog**. From here on, every query and analysis is grounded to this schema — no rows ever
leak into a model prompt.

In [ ]:
SALES = [
    {"region": "NA", "product": "alpha", "revenue": 1200.5, "units": 5},
    {"region": "EU", "product": "alpha", "revenue": 980.0, "units": 4},
    {"region": "NA", "product": "beta", "revenue": 300.0, "units": 2},
    {"region": "APAC", "product": "beta", "revenue": 1500.25, "units": 8},
]

app.register_dataset(SALES, columns=["region", "product", "revenue", "units"], name="sales")
print(f"registered 'sales' with {len(SALES)} rows")

## (A) `app.query_data` — a question grounded to read-only SQL, cited to the cell

The offline planner turns the question into a **read-only `SELECT`** (a generated write, DDL, or
injection signal is structurally refused). The result carries `rows`, **per-cell citations**
(`cite_refs`), and `verify(catalog)`, which re-derives the answer and every cited cell from the
source bytes.

In [ ]:
result = app.query_data("total units by region", table="sales")

print("rows:")
for row in result.rows:
    print(f"   {row}")

# Each result row points back at the exact source cells it was computed from.
catalog = app.data_catalog()
print(f"\nrow 0 cites: {result.cite_refs(0)}")
print(f"verifies against the source: {result.verify(catalog)}")

### Verification catches a tampered source

`verify` is not a checksum of the answer — it re-executes the query against the catalog and checks
every cited cell. Poke a source value and the same result no longer verifies.

In [ ]:
from vincio.data import DataCatalog, Dataset

tampered = DataCatalog.of(
    Dataset.from_records([{**r, "units": r["units"] + 1} for r in SALES], name="sales"),
    name="sales",
)
print(f"against a tampered source, verify() = {result.verify(tampered)} (caught)")

## (B) `app.analyze_data` — a bounded, multi-step analysis becomes a cited narrative

A real analytical question is rarely one query. `analyze_data` plans, queries, inspects, and refines
over the same read-only query plane, so every finding is grounded by construction. The whole
narrative cites the exact cells behind each statement and `verify(catalog)`s offline.

In [ ]:
analysis = app.analyze_data("how does revenue break down by region?", table="sales")

print(analysis.summary())
for step in analysis.steps:
    print(f"   - [{step.kind}] {step.finding}")

print(f"\ncoverage: {analysis.coverage}")
print(f"narrative verifies against the source: {analysis.verify(catalog)}")

## (C) `app.generate_chart` — a cited result becomes a trustworthy artifact

An analytical answer ships as a deliverable. `generate_chart` turns a cited query result into a
spec-driven **Vega-Lite v5** chart (a PNG via `matplotlib` behind the `vincio[charts]` extra). The
chart is **content-bound** (a C2PA data-driven credential binds its bytes) and **data-bound** (it
back-references the exact source cells and `verify(catalog)`s offline).

In [ ]:
result = app.query_data(
    "SELECT region, SUM(revenue) AS s FROM sales GROUP BY region ORDER BY s DESC",
    table="sales",
)
chart = app.generate_chart(result, title="Total revenue by region")

# The Vega-Lite spec exists and is portable JSON.
spec = chart.to_vega_lite()
print(f"renderer={chart.renderer}  media={chart.media_type}  points={chart.point_count}")
print(f"encoding: x={spec['encoding']['x']['field']}  y={spec['encoding']['y']['field']}")

# The chart cites its source cells and re-derives from the bytes.
print(f"\ncites: {chart.cite_refs()[:4]} ...")
print(f"chart verifies against the source: {chart.verify(app.data_catalog())}")

## Recap

A single data question travelled the whole plane: it was **planned** to read-only SQL, **executed**
where the data lives, **cited** to the exact cell, **explored** into a bounded narrative, and shipped
as a **content-bound, data-bound chart** — and every step `verify`s offline against the source
bytes. Nothing here touched a database, a network, or an API key.

Next steps: explore the governed text-to-query, data-analysis-agent, and charts examples
(`examples/15`–`17`) for the full standalone API, including injection refusal and cited reports.